# 3.5 Exercise: what did that capacity buy on the heartbeats?

Notebook 3.1 put a mixture of experts next to a dense head that uses as many parameters per
image and found that, on four kinds of photo, it won while its router was cheap: the router
gave groups of classes experts of their own.
Notebook 3.2 drew what parameters, data and compute buy as curves with their spread.
Here both come together on lesson 2's heartbeats: you add routing to a design of a beat,
put its capacity-matched control next to it, and measure a capacity curve over two axes.
At the end you can say, from your own runs, on which axis this problem is bound, what
your runs can and cannot say about whether routing earned its parameters, and what the
router did with its experts; and you can argue a search space from curves you measured.

## What is asked of you

1. **Part 1: run this notebook as it is.** The baseline; a capacity curve over
   parameters and one over data, each with the route stage and its control; what the
   router did with its experts; and an example search, whose space you hold against
   your own curves. You change nothing; you read what comes out and answer the
   questions for yourself. Part 1 runs before your own study: its search measures what
   one trial costs on your machine, and `make study LESSON=3` computes its budget from
   that.
2. **Part 2: one study of your own**, in the folder [`exercises/`](exercises/): one idea,
   stated as a hypothesis before you run, a design that can test it, a search space your
   curves argue for, the route stage paired with its control, and the use of its
   experts read afterwards. The class spreads over the open questions at the end of
   this notebook, and builds on each other's ideas.
3. **Hand in your runs** before the next lesson, with one command: `make share LESSON=3`.

## Where your runs go

As in lessons 1 and 2: every run is written to `artefacts/optuna/studies.db` on your
machine, `make optuna-dashboard` shows it, and `make share LESSON=3` hands in the runs of
lesson 3. Part 1 writes to two studies, `lesson3-notebook` (the trials of the search) and
`lesson3-notebook-runs` (the baseline and every point of the two curves). Part 2 writes to
`lesson3-study-<your study>` and its `-runs`.

**Every run of Part 1 starts from a seed.** Run r of a point starts from `SEED + r`
(`SEED` is 0), and the search of E4 draws its trials and the seeds of its pairs from
`SEED`. The two arms at one point start from the same seeds, and the design builds the
route stage or its control last, so they share every other first weight. Every run is
stored in `artefacts/lesson3/part1-seed0.json`, and the tables and figures are drawn
from that file. On the same machine and library versions a rerun gives the same numbers;
on another machine the arithmetic can differ in the last digits, and runs drift apart
from there.

The readings in this notebook are of seed 0. Three runs per point are three draws of the
seed, and `SEED = 1` gives three others. Every difference is read in points of macro
recall, beside the spread of the runs it compares: the standard deviation with `ddof=1`
(divided by `n - 1`) of each arm, and the standard error of the difference that the
tables print next to it. Where the curves per epoch still move at the last epoch, longer
training is what might settle a difference. Every spread is over seeds on the same six
validation patients: it says what other random weights could change on these six
people, not whether a difference carries over to other hearts.

## The problem, and the three axes of capacity

The data is lesson 2's: single beats from the MIT-BIH Arrhythmia Database, three classes
(N, S, V), split by patient, scored on six patients of train that no model trains on
(notebook 2.4). The metric is again **macro recall**, the mean of the recalls of N, S and
V, so "always N" scores 1/3.

A run spends capacity on three axes, and the harness of this lesson lets you turn each:

| axis | what you turn | what a run logs |
|---|---|---|
| parameters | `width` (channels) and `depth` (blocks) of the design | `n_params` |
| data | `train_fraction`: the share of the training beats a run trains on | `n_train` |
| compute | `epochs`, the passes over those beats | `train_examples`, `train_steps` |

Compute is counted here as the **training examples a run sees**, epochs × training
beats, and as its optimizer steps. In the sense of the scaling laws, compute is about
parameters × examples: a run of width 64 does many times the arithmetic of one of width
8 on the same examples. This notebook keeps the two apart, so "compute" below means
examples seen.

`train_fraction` is taken **stratified**: the beats of every class are shuffled once, with
a fixed seed, and a fraction keeps the first part of every class. A run on 0.2 trains on
the same mix of N, S and V as a run on 1.0, and on a subset of the beats of a run on 0.5.
What moves with it is compute: at the same `epochs`, a run on 0.2 of the beats sees a
fifth of the examples and takes a fifth of the optimizer steps of a run on all of them.
E3 therefore measures the data axis twice, at equal epochs and at equal steps
(`equal_steps`, which passes over the smaller set about `1 / train_fraction` times per
epoch).

Every run trains `EPOCHS` epochs (30 here) at one learning rate, 1e-3, for every arm and
every width. E4 draws the curve per epoch, so you can check whether the arms had stopped
climbing when they were compared.

## Setup

The data, the trainer and the design come from the folder `exercises/`, the same code
your own study uses in Part 2:

| name | what it is |
|---|---|
| `HARNESS.data(config)` | the train loader (the `train_fraction` of the training beats) and the validation loader |
| `HARNESS.fit(model, data, config)` | trains a model and scores it on the validation patients after every epoch |
| `DESIGN.build(config, n_classes=3)` | the design of [`exercises/designs.py`](exercises/designs.py): lesson 2's `Beats` with the options `route` and `matched` |
| `spread(config, name)` | `REPEATS` runs of one config, run r from the seed `SEED + r`, logged to `RUNS`: the objective per run, its recall per class, `n_params`, the objective after every epoch, and the use of the experts |
| `against(runs)`, `steps(runs)` | per point of a curve, route against matched; per step from one point to the next, what it gained; both in points, beside the standard error of the difference |
| `expert_use(model)` | for a trained model with a router, per expert: the share of the validation positions it leads, its mean router weight (notebook 3.1 counts the first, in 3.1.7), and the peak of both at one position or in one class |
| `search(space, name)` | `N_TRIALS` trials of a search space as one Optuna study; every draw of the route stage is followed by its twin with `matched` flipped, on the same seed (`paired`) |
| `pairs(table)`, `paired_curve(pairs, axis)` | the pairs of a search, and along one axis per group of pairs: the number of pairs, each arm's mean, the mean difference within the pairs, and its standard error, paired and unpaired (E4 explains both) |
| `RUNS` | the study `lesson3-notebook-runs`; `RUNS.log(...)` stores one run in it |

In [ ]:
import logging

# on macOS, torch warns about redirects when vectormesh loads it; the warning is about
# torch's distributed launcher, which this notebook does not use
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(
    logging.ERROR
)

In [ ]:
import io
import itertools
import json
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from exercises.designs import DESIGN, matched_hidden
from exercises.harness import (
    CHOICES,
    CLASS_DB,
    CLASSES,
    HARNESS,
    IDLE,
    LINEAR,
    TRAIN_ROWS,
    Use,
    arm,
    class_pairs,
    expert_use,
    fraction,
    groups,
    paired,
    paired_curve,
    pairs,
    seconds_per_trial,
    splits,
    trial_rows,
    use_attrs,
    use_table,
)
from huggingface_hub.utils import disable_progress_bars as hub_quiet
from loguru import logger
from matplotlib.ticker import NullFormatter
from notebooktester import param
from torch import nn

from hpo.measure import Architecture, Confusion, repeat
from hpo.record import Idea, RunRecord, Store, StudyLog
from hpo.search import Budget, Choice, Float, Int, Space

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
hub_quiet()
optuna.logging.set_verbosity(optuna.logging.WARNING)

EPOCHS = HARNESS.epochs
REPEATS = HARNESS.repeats
N_TRIALS = param(24, test=2)
WIDTHS = param([8, 16, 32, 64], test=[8, 16])  # the parameter axis of E2
FRACTIONS = param([0.05, 0.2], test=[0.2])  # the data axis of E3, besides 1.0
OBJECTIVE = "macro_recall"
# Every run of Part 1 starts from a seed: run r of a point from SEED + r, the search from
# SEED. On the same machine and library versions a rerun gives the same numbers; another
# SEED gives other runs, and the readings below say what that could change.
SEED = 0
# every run of Part 1 is stored here, and the tables and figures are drawn from this file
TESTING = Store.default().testing
RESULTS = (
    Store.default().root.parent
    / "lesson3"
    / f"part1-seed{SEED}{'-test' if TESTING else ''}.json"
)
EXAMPLE = torch.zeros(2, 187, 1)  # two beats are enough to trace the shapes

# Part 1 is the same for everyone, so its idea is the course's: "lesson 3 as handed out".
HANDED_OUT = Idea.handed_out(lesson=3)
RUNS = StudyLog.runs(
    name="lesson3-notebook-runs", idea=HANDED_OUT, objective=OBJECTIVE, choices=CHOICES
)

# the trainer's settings of every run in Part 1
TRAINER = {
    "dataset": "mitbih",
    "epochs": EPOCHS,
    "lr": 1e-3,
    "class_weight": 0.0,
    "train_fraction": 1.0,
}
# the handed-out design: every lesson 2 option off, the route stage on
ROUTE = {**vars(DESIGN.config()), **TRAINER, "route": True, "matched": False}
MATCHED = {**ROUTE, "matched": True}
ROLE = {"route": "hypothesis", "matched": "ablation"}


def n_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


@dataclass
class Spread:
    values: np.ndarray  # the objective of every repeat
    recall: dict[str, np.ndarray]  # per class, the recall of every repeat
    n_params: int
    curves: list[list[float]]  # per repeat, the objective after every epoch
    use: list[Use]  # per repeat, the use of the route stage's experts (empty without)
    seeds: list[int]  # per repeat, the seed it started from


def spread(config: dict, name: str) -> Spread:
    # `REPEATS` runs of the design `config` chooses, run r from the seed SEED + r,
    # logged to RUNS under `name` with the use of their experts
    data = HARNESS.data(config)
    make = lambda: DESIGN.build(config, n_classes=len(data.classes))
    fits, uses = [], []

    def fit(model: nn.Module):
        fits.append(HARNESS.fit(model, data, config))
        uses.append(expert_use(model))  # after training, on the validation patients
        return fits[-1]

    seeds = list(range(SEED, SEED + REPEATS))
    queue = iter(seeds)

    def seeded() -> nn.Module:
        # run r starts from SEED + r: its first weights and the order it sees the beats
        # in; every point of a curve uses the same seeds
        seed = next(queue)
        torch.manual_seed(seed)
        model = make()
        torch.manual_seed(seed)
        return model

    runs = repeat(seeded, fit, REPEATS)
    RUNS.log(
        name=name,
        role=ROLE[arm(config)],
        results=runs,
        dataset="mitbih",
        params=HARNESS.params(config, data),
        architecture=Architecture.of(make(), EXAMPLE),
        attrs=use_attrs(uses),
    )
    return Spread(
        values=runs.values(OBJECTIVE),
        recall={c: runs.values(f"recall/{c}") for c in CLASSES},
        n_params=n_params(make()),
        curves=[getattr(f.curves, OBJECTIVE) for f in fits],
        use=[u["experts"] for u in uses if "experts" in u],
        seeds=seeds,
    )


def save(experiment: str, spreads: dict) -> None:
    # every run of `experiment` into RESULTS, keyed by its point, next to the other
    # experiments already there
    stored = json.loads(RESULTS.read_text()) if RESULTS.exists() else {}
    stored[experiment] = [
        {
            "point": list(point),
            "values": s.values.tolist(),
            "recall": {c: v.tolist() for c, v in s.recall.items()},
            "n_params": s.n_params,
            "curves": s.curves,
            "use": [asdict(u) for u in s.use],
            "seeds": s.seeds,
        }
        for point, s in spreads.items()
    ]
    RESULTS.parent.mkdir(parents=True, exist_ok=True)
    RESULTS.write_text(json.dumps(stored))


def load(experiment: str) -> dict:
    # the runs of `experiment` as RESULTS holds them, point -> Spread
    return {
        tuple(r["point"]): Spread(
            values=np.array(r["values"]),
            recall={c: np.array(v) for c, v in r["recall"].items()},
            n_params=r["n_params"],
            curves=r["curves"],
            use=[Use(**u) for u in r["use"]],
            seeds=r["seeds"],
        )
        for r in json.loads(RESULTS.read_text())[experiment]
    }


def objective(space: Space, log: StudyLog, remaining: list[int]):
    # what an Optuna study calls once per trial: draw a config, build the design from the
    # pair's seed, train it, report the macro recall after every epoch, and return the
    # last one; `paired` queues the twin of every route draw with `matched` flipped
    seed: list[int | None] = [None]

    def seeded(value: int | None) -> None:
        seed[0] = value

    def run_trial(trial: optuna.Trial) -> float:
        remaining[0] -= 1
        config = space.draw(trial)
        data = HARNESS.data(config)
        torch.manual_seed(seed[0])
        model = DESIGN.build(config, n_classes=len(data.classes))
        torch.manual_seed(seed[0])  # both arms of a pair see the beats in one order
        seen = None

        def to_optuna(epoch, fit):
            nonlocal seen
            seen = fit
            trial.report(fit.scores().get(OBJECTIVE), epoch)

        try:
            seen = HARNESS.fit(model, data, config, to_optuna)
            return seen.scores().get(OBJECTIVE)
        finally:
            log.record(
                trial,
                RunRecord(
                    role="trial",
                    dataset="mitbih",
                    config=config,
                    fixed=space.fixed(),
                    train_seconds=seen.seconds if seen else 0.0,
                    curves=seen.curves if seen else None,
                    scores=seen.scores() if seen and seen.confusion else None,
                    confusion=seen.confusion if seen else None,
                    extra=dict(data.dims),
                ),
                Architecture.of(model, EXAMPLE),
            )
            # the use of the experts, after training, as user attrs of the trial
            for key, value in use_attrs([expert_use(model)]).items():
                trial.set_user_attr(key, value)

    return paired(
        run_trial,
        log.study,
        seeded,
        lambda: remaining[0],
        seeds=np.random.default_rng(SEED),
    )


def search(space: Space, name: str, n_trials: int = N_TRIALS) -> StudyLog:
    # `n_trials` trials of `space` as the Optuna study `name`: random draws from SEED,
    # each draw of the route stage followed by its twin; the study starts empty, so a
    # rerun replaces the trials of the last run instead of adding to them
    store = Store.default()
    if name in optuna.get_all_study_names(store.url):
        optuna.delete_study(study_name=name, storage=store.url)
    study = StudyLog.search(
        name=name,
        space=space,
        idea=HANDED_OUT,
        objective=OBJECTIVE,
        attrs={"origin": "notebook", "seed": SEED},
        sampler=optuna.samplers.RandomSampler(seed=SEED),
    )
    start = time.time()
    study.optimize(objective(space, study, [n_trials]), n_trials=n_trials)
    done = study.trials(optuna.trial.TrialState.COMPLETE)
    print(
        f"{n_trials} trials in {time.time() - start:.0f}s; {len(done)} finished in all"
    )
    return study


def log_axis(ax, ticks) -> None:
    # a log x axis labelled at the points of the curve, without the minor labels
    ticks = sorted(set(ticks))
    ax.set_xscale("log")
    ax.set_xticks(ticks, [f"{t:,}" for t in ticks])
    ax.xaxis.set_minor_formatter(NullFormatter())


def plot_curve(x: dict, runs: dict, xlabel: str, title: str, ax=None) -> None:
    # one line per arm: the mean over the repeats with its spread (ddof=1), every
    # repeat a dot
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 3.6))
    for name, color in [("route", "tab:orange"), ("matched", "tab:blue")]:
        xs = np.array([x[name, point] for point in runs[name]])
        values = list(runs[name].values())
        means = np.array([v.mean() for v in values])
        stds = np.array([v.std(ddof=1) for v in values])
        ax.errorbar(xs, means, yerr=stds, fmt="o-", color=color, capsize=4, label=name)
        for xv, v in zip(xs, values, strict=True):
            ax.scatter([xv] * len(v), v, color=color, alpha=0.3, s=14)
    log_axis(ax, [x["route", point] for point in runs["route"]])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"{OBJECTIVE} on the validation patients")
    ax.set_title(title, loc="left", fontsize=10)
    ax.legend(title="arm")
    ax.spines[["top", "right"]].set_visible(False)
    if show:
        plt.show()


def se_of(a: np.ndarray, b: np.ndarray) -> float:
    # lesson 2's standard error of the difference of two independent means, with the
    # ddof=1 spreads: sqrt(std_a² / n_a + std_b² / n_b)
    return float(np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b)))


def against(runs: dict, label: str) -> pd.DataFrame:
    # per point of a curve: the runs per arm, both arms' mean and spread (ddof=1), and
    # route against matched in points, beside the standard error of that difference
    rows = {}
    for point in runs["route"]:
        route, matched = runs["route"][point], runs["matched"][point]
        rows[f"{point:g}"] = {
            "n": len(route),
            "route": route.mean(),
            "route std": route.std(ddof=1),
            "matched": matched.mean(),
            "matched std": matched.std(ddof=1),
            "route - matched (pt)": 100 * (route.mean() - matched.mean()),
            "se (pt)": 100 * se_of(route, matched),
        }
    table = pd.DataFrame(rows).T
    table.index.name = label
    return table


def steps(runs: dict, label: str) -> pd.DataFrame:
    # per arm, every step from one point of a curve to the next: what the mean gained in
    # points, beside the standard error of the difference of the two points
    rows = []
    for name in ["route", "matched"]:
        points = list(runs[name])
        for lo, hi in itertools.pairwise(points):
            rows.append(
                {
                    "arm": name,
                    label: f"{lo} to {hi}",
                    "gain (pt)": 100 * (runs[name][hi].mean() - runs[name][lo].mean()),
                    "se (pt)": 100 * se_of(runs[name][hi], runs[name][lo]),
                }
            )
    return pd.DataFrame(rows).set_index(["arm", label])


def show(table: pd.DataFrame) -> None:
    # a table of `against` or `steps`: three decimals for scores, one for points
    scores = [c for c in ["route", "route std", "matched", "matched std"] if c in table]
    points = [c for c in ["route - matched (pt)", "gain (pt)"] if c in table]
    display(
        table.style.format("{:.3f}", subset=scores)
        .format("{:+.1f}", subset=points, na_rep="-")
        .format("{:.1f}", subset=["se (pt)"], na_rep="-")
    )


train, valid = splits(TRAIN_ROWS)
counts = pd.DataFrame(
    {"train": train.counts(), "validation": valid.counts()}, index=CLASSES
).T
display(counts.assign(beats=counts.sum(axis=1)))

The S column is the one to keep in mind: S is about one beat in seventy in train. A run
on a fraction of the beats trains on that fraction of the S beats too.

# Part 1: the notebook as handed out

## E1: the baseline, first

The floors of notebook 2.4: always N, and a linear model on the 187 steps, repeated so
you know its spread. The linear model is the baseline every later number is read
against, here and in your own study.

In [ ]:
always_n = Confusion.of(valid.labels, torch.zeros_like(valid.labels), CLASSES)
linear_config = {**LINEAR, **TRAINER}
linear_data = HARNESS.data(linear_config)
make_linear = lambda: nn.Sequential(nn.Flatten(), nn.Linear(187, 3))
linear_runs = repeat(
    make_linear, lambda model: HARNESS.fit(model, linear_data, linear_config), REPEATS
)
baseline = linear_runs.values(OBJECTIVE)
RUNS.log(
    name="baseline",
    role="baseline",
    results=linear_runs,
    dataset="mitbih",
    params=HARNESS.params(linear_config, linear_data),
    architecture=Architecture.of(make_linear(), EXAMPLE),
)
print(f"always N           macro recall {always_n.scores().macro_recall:.3f}")
print(
    f"linear, {REPEATS} runs    macro recall {baseline.mean():.4f}, "
    f"std {baseline.std(ddof=1):.4f}"
)

## E2: the route stage, its control, and a curve over parameters

The design of [`exercises/designs.py`](exercises/designs.py) is lesson 2's `Beats`: a
kernel of 5, three blocks of `width` channels, each followed by a pool of 2, and a
linear head. Lesson 3 adds one stage after the last block, on the shortened beat of 23
positions, and its control:

<svg width="640" height="150" viewBox="0 0 640 150" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="12">
  <text x="10" y="20" font-weight="bold">route</text>
  <rect x="10" y="30" width="120" height="34" fill="#eee" stroke="#555"/>
  <text x="20" y="52">blocks, pooled</text>
  <text x="140" y="44">(B, 23, w)</text>
  <line x1="130" y1="47" x2="230" y2="47" stroke="#555" marker-end="url(#a)"/>
  <rect x="230" y="26" width="200" height="42" fill="#fde5cc" stroke="#c60"/>
  <text x="240" y="44">router picks or weighs</text>
  <text x="240" y="60">n_experts experts, w -&gt; w</text>
  <line x1="430" y1="47" x2="520" y2="47" stroke="#555" marker-end="url(#a)"/>
  <text x="440" y="44">(B, 23, w)</text>
  <rect x="520" y="30" width="80" height="34" fill="#eee" stroke="#555"/>
  <text x="535" y="52">head</text>
  <text x="10" y="100" font-weight="bold">matched</text>
  <rect x="10" y="110" width="120" height="34" fill="#eee" stroke="#555"/>
  <text x="20" y="132">blocks, pooled</text>
  <text x="140" y="124">(B, 23, w)</text>
  <line x1="130" y1="127" x2="230" y2="127" stroke="#555" marker-end="url(#a)"/>
  <rect x="230" y="106" width="200" height="42" fill="#d6e6f5" stroke="#36c"/>
  <text x="240" y="124">Linear w -&gt; h, GELU, Linear h -&gt; w</text>
  <text x="240" y="140">h chosen for the same n_params</text>
  <line x1="430" y1="127" x2="520" y2="127" stroke="#555" marker-end="url(#a)"/>
  <text x="440" y="124">(B, 23, w)</text>
  <rect x="520" y="110" width="80" height="34" fill="#eee" stroke="#555"/>
  <text x="535" y="132">head</text>
  <defs><marker id="a" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 z" fill="#555"/></marker></defs>
</svg>

- **`route`** is notebook 3.1's mixture of experts, on every one of the 23 positions:
  `n_experts` experts (4 here), each a linear layer of `w -> w` with a GELU, and a router
  that reads the position and weighs the experts' outputs with a softmax (`MoE`, dense):
  every expert runs at every position.
  The shape stays `(B, 23, w)`.
- **`matched`** is its capacity-matched control: the route stage replaced by
  one plain path, `w -> h -> w`, with `h` chosen so the stage has the same number of
  parameters. No router, no experts; the same path at every position.

The two arms differ in what they do with the same parameters: weigh four experts of
`w -> w` per position, or spend them on one path `w -> h -> w`, with `h` about twice `w`.
They also differ in the form of the function: every expert ends in a GELU and the
mixture feeds the head directly, while the control's last layer is linear. The
parameter count is what is matched, not the shape of the function. Both arms train at
the same learning rate, 1e-3; a router and a plain path need not share their best one,
so every comparison here is one at that learning rate. Read it in atlas
before you run it:

```bash
atlas-code-nightly view notebooks/3_scaling_and_search/exercises
```

The cell below builds both arms at every width of the curve and counts their
parameters. **Check that they match** before anything is trained: a curve whose arms
differ in size answers another question.

In [ ]:
sizes = pd.DataFrame(
    {
        w: {
            "route": n_params(DESIGN.build({**ROUTE, "width": w}, n_classes=3)),
            "matched": n_params(DESIGN.build({**MATCHED, "width": w}, n_classes=3)),
            "h of matched": matched_hidden(w, ROUTE["n_experts"]),
        }
        for w in WIDTHS
    }
).T
sizes["difference"] = (sizes["matched"] - sizes["route"]) / sizes["route"]
sizes.index.name = "width"
display(sizes.style.format({"difference": "{:+.2%}"}))

The curve: both arms at every width, on every training beat, `REPEATS` runs each, for
`EPOCHS` epochs. The width doubles from one point to the next, so the parameters grow
three to four times per step (the table above: from 1,911 to 6,247 is x3.3, from 22,215
to 83,335 x3.8).

**Before you run the next cell**, write down two things: at which width you expect the
curve to flatten, and whether you expect the route arm above its control, and by how
many points. Notebook 3.1 found a gain for routing on four kinds of photo, where the
experts split the classes between them. Here the router reads each of the 23 positions
of a beat on its own, so it could give the rise of a beat one expert and its peak
another; whether that pays is what the curve answers.

In [ ]:
e2 = {}  # (arm, width) -> Spread
for w in WIDTHS:
    for name, config in [("route", ROUTE), ("matched", MATCHED)]:
        e2[name, w] = spread({**config, "width": w}, f"{name}, width {w}")
save("e2", e2)

# from here on, every number is read from the stored runs
e2 = load("e2")
by_width = {
    name: {w: e2[name, w].values for w in WIDTHS} for name in ["route", "matched"]
}
x_params = {(name, w): e2[name, w].n_params for name, w in e2}
plot_curve(
    x_params,
    by_width,
    "parameters of the model (n_params)",
    f"the curve over parameters, on every training beat, {REPEATS} runs per point",
)
print("route against matched, per width")
show(against(by_width, "width"))
print("the steps of the curve: from one width to the next, per arm")
show(steps(by_width, "width"))
# the recall of every class, once: the loss weighs no class (class_weight 0)
recall = pd.DataFrame(
    {
        (name, w): {c: e2[name, w].recall[c].mean() for c in CLASSES}
        for name in ["route", "matched"]
        for w in WIDTHS
    }
).T
recall.index.names = ["arm", "width"]
print(f"recall per class, mean of {REPEATS} runs")
display(recall.round(3))

Read the curve and the tables:

1. **Where does the curve flatten?** Read the table of steps: per arm, what the mean
   gained from one width to the next, in points, with the standard error of that gain
   beside it. Where the gains are about as large as their standard errors, other seeds
   could reorder the steps: that stretch is open, not flat.
2. **Did routing beat its control?** The column *route - matched (pt)* of the first
   table is the difference of the two arms, with its standard error beside it. Does the
   sign hold from one width to the next? If it does not, the route stage bought what its
   parameters bought, and nothing more.
3. **Had both arms stopped climbing?** The comparison is at `EPOCHS` epochs, on the
   score of the last epoch. If one arm still climbs at the last epoch (E4 draws the curve
   per epoch), a difference is partly one of learning speed, not of what the arm can
   reach.
4. **Which class moves?** The loss weighs every class alike (`class_weight` 0), and S is
   one beat in seventy. Look at the recall per class: is the macro recall of these runs
   a recall of N, S and V, or mostly of N and V?

Nothing in Part 1 is seeded, so your numbers will differ; the readings below are of the run they were written from.

**Three runs per point do not show where the curve over parameters flattens.** The route
arm gains +1.9 points from width 8 to 16 (se 1.4), then +0.4 and -0.8; the control gains
+1.4 (se 1.4), +2.3 (se 1.2) and +0.7. The control's mean rises at every step and the
route arm's stops rising after width 16, but every step is about the size of its
standard error, so other seeds could reorder them: the whole range from 8 to 64 stays
open.

**Routing did not beat its control.** The route arm is below `matched` at every width:
by 0.6, 0.1, 2.1 and 3.5 points. The standard error of the difference is about 1.5
points at width 8 and 1.3 at width 32; at width 64 it is 0.6, against a gap of 3.5. Two
things make that gap less firm than it looks. Its standard error leans on a control
whose three runs agree to 0.004 (the route arm's spread is 0.010), and three runs say
little about how wide a spread is: a spread that small can be luck. And the compute cell
below shows the gap at width 64 is 1.3 points averaged over the last ten epochs, not
3.5. What this supports: at these widths and at this learning rate the route stage
bought nothing the same parameters spent on a plain path did not; at the widest, less.

**What the macro recall is made of.** The recall of S is 0.02 to 0.05 in every arm and at
every width, against 0.91 to 0.98 for N and 0.64 to 0.75 for V: with the plain loss
these models find almost no S beat, so the macro recall moves with N and V. Along width
the two arms grow in different classes. From width 8 to 64 the route arm gains 5.2
points of N and loses 0.5 of V; the control gains 6.4 of N and 5.5 of V. Between the
arms, V is where the route arm falls behind: by 3.1, 4.3, 7.1 and 9.1 points at widths 8
to 64, while their N differs by -1.3 to +3.3 points with no direction. These are means
of three runs, printed without their spread, so they say where to look: a study that
asks why routing falls behind can read the recall of V of its pairs.

The linear baseline (0.355) sits just above always N (0.333): it marks the floor, and
every model of the curve is far above it; it does not tell the designs apart.

### What did the router do with its experts?

A score says how well the route arm did, not whether it routed. Notebook 3.1 (3.1.7)
counted where a router sends its inputs. The slides of lesson 3 showed **router
collapse**: after training, every input goes to one expert, and the model is a network
with a single expert. The cell below makes the same
count here, on every one of the 23 positions of
every beat of the validation patients, for each run of the route arm of E2. It gives
three numbers per expert:

- **share**: the share of the positions whose router gives this expert its largest
  weight. A sparse top-1 router (`SparseMoE`) runs only that expert, so for sparse
  routing the share is what the experts get.
- **weight**: the router's softmax weight for this expert, averaged over the positions.
  The route arm here is dense (`MoE`): it mixes every expert into every position with
  these weights, so for dense routing the weight says how much of each expert the router
  mixes in. (What reaches the output is that weight times the expert's output, and an
  expert can learn a large or a small output.) An expert can lead at every position (a
  share of 1) with a weight of 0.3: that is a mix in which one expert is a little ahead,
  not a collapse.
- **peak**: the measure that fits the router (weight for dense, share for sparse),
  averaged over the beats at one of the 23 positions, or over the beats of one class;
  the peak is the largest of those 23 + 3 averages. A mean
  over every position cannot tell an expert that is never used from a specialist: an
  expert that takes the same one of the 23 positions in every beat, say the one of the R
  peak, has a share of 1/23 = 0.043 overall and a peak of 1, and one that takes the S
  beats, 3% of the validation beats, has a peak of 1 too.

Even use is `1 / n_experts`, 0.25 for four experts. Words for what lies between:

- an expert is **idle** when its peak is below 5% (`IDLE`): at no position and in no
  class does it get 5%;
- a router with one expert in use is **collapsed**;
- a router with idle experts and more than one in use is **partly collapsed**;
- a router with no idle expert is **spread**.

The bars are in the order of the experts in all three rows, so within a run expert 2
above is expert 2 below. Across runs the numbers mean nothing: every run starts from
its own seed, and which expert ends up with which positions falls out of
training, so expert 2 of one run has nothing to do with expert 2 of the next.

**Before you run the next cell**, write down what you expect: do the four experts share
the positions, or does one take them, and does the answer change with the width?

In [ ]:
N_EXPERTS = ROUTE["n_experts"]
use = pd.DataFrame(
    [
        {
            "width": w,
            "run": r + 1,
            OBJECTIVE: e2["route", w].values[r],
            "in use": u.in_use,
            "state": u.state,
            "top weight": max(u.weight),
            "weight per expert": " ".join(f"{v:.3f}" for v in u.weight),
            "share per expert": " ".join(f"{s:.3f}" for s in u.share),
            "peak per expert": " ".join(f"{p:.3f}" for p in u.peak),
        }
        for w in WIDTHS
        for r, u in enumerate(e2["route", w].use)
    ]
).set_index(["width", "run"])
display(use.style.format("{:.3f}", subset=[OBJECTIVE, "top weight"]))

ROWS = {
    "weight": "mean weight",
    "share": "share of the positions",
    "peak": "peak weight, at one\nposition or in one class",
}
fig, panels = plt.subplots(
    len(ROWS), len(WIDTHS), figsize=(3.2 * len(WIDTHS), 7), sharey=True, squeeze=False
)
offset = 0.8 / REPEATS
for col, w in enumerate(WIDTHS):
    for row, kind in enumerate(ROWS):
        ax = panels[row, col]
        for r, u in enumerate(e2["route", w].use):
            ax.bar(
                np.arange(N_EXPERTS) + (r - (REPEATS - 1) / 2) * offset,
                getattr(u, kind),
                width=offset,
                color=f"C{r}",
                label=f"run {r + 1}",
            )
        ax.axhline(1 / N_EXPERTS, color="gray", ls="--", lw=1)
        ax.axhline(IDLE, color="tab:red", ls=":", lw=1)
        ax.set_xticks(range(N_EXPERTS), [str(i + 1) for i in range(N_EXPERTS)])
        ax.spines[["top", "right"]].set_visible(False)
        if row == 0:
            ax.set_title(f"width {w}", fontsize=10)
        if row == len(ROWS) - 1:
            ax.set_xlabel("expert")
        if col == 0:
            ax.set_ylabel(ROWS[kind], fontsize=9)
panels[0, -1].legend(fontsize=8)
fig.suptitle(
    "the experts of every route run of E2, on the validation patients, in expert order "
    "(dashed: even use; dotted: 5%; an expert whose peak is below it is idle)",
    x=0.02,
    ha="left",
    fontsize=10,
)
plt.tight_layout()
plt.show()

Nothing in Part 1 is seeded, so your numbers will differ; the readings below are of the run they were written from.

**No router collapsed; one partly did, at width 8.** Of the twelve route runs, eleven
are spread and one, width 8 run 3, is partly collapsed: its expert 1 gets a mean weight
of 0.025 and a peak that prints as 0.050. The count compares the unrounded peak with 5%,
and this one is just under it, so the expert counts as idle and three experts are in
use. A label decided on the fourth decimal says nothing about the router: read this run
as spread at its edge, not as a router that gave up an expert. The most weighted expert
gets 0.51 to 0.61 at width 8, 0.48 to 0.65 at 16, 0.39 to 0.57 at 32 and 0.45 to 0.54 at
64, against 0.25 for even use.

**The peak row shows what the mean hides.** At width 16, expert 4 of run 1 gets a mean
weight of 0.046 and expert 3 of run 2 one of 0.047: below 5%, so a rule on the mean would
call them idle. Their peaks are 0.115 and 0.127: at some position, or in some class, the
router gives them more than twice that. And several experts reach a peak near 1 (0.95 to
1.00, for example expert 2 of width 32 run 1, mean 0.568 and peak 0.999): at one
position or in one class the router gives that expert almost all the weight. The table
does not say which position or which class; that is a question a study can ask.

The share row shows how a dense mix hides in a count: in width 8 run 2, experts 2 and 3
lead at no position (a share of 0.000) and still get weights of 0.062 and 0.075, so they
are mixed into every output.

Concentration does not line up with the score here: the partly collapsed run of width 8
scores 0.542, the two spread ones 0.510 and 0.541. With three runs per width that is no
evidence either way.

Two questions for you. What would a collapsed router look like in these three rows of
bars, and what would its stage then be? (A dense mixture with weight 1 on one expert is
that expert alone, `w -> w`, a quarter of the stage's parameters.) And what is a
load-balancing loss for (the slides of lesson 3)? Here `balance` changes nothing, because
the route arm is dense and only
`SparseMoE` computes that loss; it is an axis for `routing="sparse"` in Part 2.

## E3: a curve over data, at equal epochs and at equal steps

The same two arms at the middle width, 32, trained on a share of the beats. A smaller
share is less data, and at the same `epochs` it is also fewer optimizer steps: a curve
at equal epochs moves data and compute together. So the curve is drawn twice:

- **equal epochs**: every share gets `EPOCHS` passes over its beats, so a smaller share
  also gets fewer steps;
- **equal steps**: every share gets about the steps of a run on every beat
  (`equal_steps`: an epoch passes over the share's beats about `1 / train_fraction`
  times), so the share changes the data and not the compute.

The points at 1.0 are E2's: the same config, so they are reused, not run again. The cell
prints, per share, the beats, the S beats among them and the steps of a run.

**Before you run the next cell**, write down what you expect: does the curve over data
flatten earlier or later than the one over parameters, does the router need more data
than its control to pay, or less, and how much of the drop at a small share is the
missing steps?

In [ ]:
MIDDLE = 32 if 32 in WIDTHS else WIDTHS[-1]
SETTINGS = {"equal epochs": False, "equal steps": True}
by_data = {s: {"route": {}, "matched": {}} for s in SETTINGS}
e3 = {}  # (setting, arm, share) -> Spread
x_data = {}
sizes_data = {}
for share in [*FRACTIONS, 1.0]:
    beats = fraction(TRAIN_ROWS, share)
    for setting, equal in SETTINGS.items():
        for name, config in [("route", ROUTE), ("matched", MATCHED)]:
            config = {**config, "width": MIDDLE, "train_fraction": share}
            config["equal_steps"] = equal
            if share == 1.0:
                e3[setting, name, share] = e2[name, MIDDLE]
            else:
                label = f"{name}, train_fraction {share}, {setting}"
                e3[setting, name, share] = spread(config, label)
            by_data[setting][name][share] = e3[setting, name, share].values
            x_data[name, share] = len(beats.labels)
            dims = HARNESS.data(config).dims
            sizes_data[share, setting] = dims["train_steps"]
    sizes_data[share] = (len(beats.labels), int(beats.counts()[1]))
save("e3", e3)

# from here on, every number is read from the stored runs
e3 = load("e3")
by_data = {
    s: {
        name: {share: e3[s, name, share].values for share in [*FRACTIONS, 1.0]}
        for name in ["route", "matched"]
    }
    for s in SETTINGS
}
print(
    f"{'share':>6} {'beats':>7} {'S beats':>8} {'steps, equal epochs':>20} {'steps, equal steps':>19}"
)
for share in [*FRACTIONS, 1.0]:
    n, s = sizes_data[share]
    print(
        f"{share:>6} {n:>7,} {s:>8,} {sizes_data[share, 'equal epochs']:>20,} "
        f"{sizes_data[share, 'equal steps']:>19,}"
    )
fig, panels = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, setting in zip(panels, SETTINGS, strict=True):
    plot_curve(x_data, by_data[setting], "training beats (n_train)", setting, ax=ax)
plt.suptitle(
    f"the curve over data, at width {MIDDLE}, {REPEATS} runs per point",
    x=0.02,
    ha="left",
    fontsize=10,
)
plt.tight_layout()
plt.show()
for setting in SETTINGS:
    print(f"{setting}: route against matched, per share")
    show(against(by_data[setting], "train_fraction"))
    print(f"{setting}: the steps of the curve, from one share to the next, per arm")
    show(steps(by_data[setting], "train_fraction"))

In [ ]:
# every share against the smallest one, per setting and arm: the gain in points, beside
# the standard error of that gain
SMALL = FRACTIONS[0]
rows = []
for setting in SETTINGS:
    for name in ["route", "matched"]:
        runs = by_data[setting][name]
        for share in [s for s in runs if s != SMALL]:
            rows.append(
                {
                    "setting": setting,
                    "arm": name,
                    "train_fraction": f"{SMALL} to {share}",
                    "gain (pt)": 100 * (runs[share].mean() - runs[SMALL].mean()),
                    "se (pt)": 100 * se_of(runs[share], runs[SMALL]),
                }
            )
print(f"every share against {SMALL}, per setting and arm")
show(pd.DataFrame(rows).set_index(["setting", "arm", "train_fraction"]))

The points above are the score of the last epoch. On the smallest share, a run at equal
steps passes over the same few beats many times: on 5% of the beats, twenty times per
epoch, six hundred times in all. If those passes helped at first and the model then
learned the few S beats by heart, the last epoch is past its best, and the point says
less about what the steps bought. The cell below draws the curve per epoch of those
runs against the optimizer steps they had taken, next to the runs on every beat.

In [ ]:
SMALL = FRACTIONS[0]
lines = [
    (f"{SMALL}, equal epochs", "equal epochs", SMALL, ":"),
    (f"{SMALL}, equal steps", "equal steps", SMALL, "-"),
    ("1.0, every beat", "equal epochs", 1.0, "--"),
]
fig, panels = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
rows = {}
for ax, name in zip(panels, ["route", "matched"], strict=True):
    for label, setting, share, style in lines:
        config = {**ROUTE, "width": MIDDLE, "train_fraction": share}
        config["equal_steps"] = SETTINGS[setting]
        per_epoch = np.array(e3[setting, name, share].curves)  # (repeats, epochs)
        per_step = HARNESS.data(config).dims["train_steps"] / per_epoch.shape[1]
        taken = per_step * np.arange(1, per_epoch.shape[1] + 1)
        ax.plot(taken, per_epoch.mean(0), style, color="k", label=label)
        best = per_epoch.mean(0).argmax()
        rows[name, label] = {
            "best epoch": best + 1,
            "best mean": per_epoch.mean(0)[best],
            "last epoch mean": per_epoch.mean(0)[-1],
            "mean, last 10 epochs": per_epoch.mean(0)[-10:].mean(),
        }
    ax.set_xscale("log")
    ax.set_xlabel("optimizer steps taken")
    ax.set_title(f"{name}, width {MIDDLE}", loc="left", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
panels[0].set_ylabel(f"{OBJECTIVE}, mean of {REPEATS} runs")
panels[-1].legend(fontsize=8)
plt.suptitle(
    "E3's runs after every epoch: the smallest share, and every beat",
    x=0.02,
    ha="left",
    fontsize=10,
)
plt.tight_layout()
plt.show()
by_steps = pd.DataFrame(rows).T
by_steps["best epoch"] = by_steps["best epoch"].astype(int)
by_steps.index.names = ["arm", "run"]
display(by_steps.round(3))

1. **What does the drop at a small share consist of?** Compare the two panels: the gap
   between equal epochs and equal steps at a share is what the missing steps cost; what
   remains at equal steps is what the missing beats cost. Read both in points, beside
   their standard errors. And check
   the curve per epoch before you trust the equal-steps point: if that run peaked early
   and fell, its last epoch is past its best, and the gap says less about steps than it
   seems to.
2. **The floor.** A macro recall of 0.333 with no spread is always N: the model says N
   for every beat, so the recall of S and of V are both 0. A point there says the runs
   learned nothing, not how much they learned.
3. **On which axis is this problem bound?** Put the curves side by side: which one still
   climbs at its right end, and which has flattened? Read the climb at equal steps, where
   the share moves the data alone. If the binding axis is at its limit (here 1.0 is every
   training beat there is), that is a finding about the budget, not a range to search
   for a better score past it. (The data curve has three points, so it can say whether it
   still climbs at 1.0, not where it bends.)
4. Did routing beat its control anywhere on this curve? By how many points, beside the
   standard error, and does the sign hold from one share to the next?

Nothing in Part 1 is seeded, so your numbers will differ; the readings below are of the run they were written from.

**On 5% of the beats, this run cannot tell what the missing steps cost.** On 1,672
beats, 23 of them S, a run at equal epochs takes 420 steps and one at equal steps 7,860.
At their last epoch, the equal-steps runs score lower, not higher: 0.485 against 0.513
for the route arm, 0.472 against 0.483 for the control. The table under the curves per
epoch says why the two do not compare: the equal-steps runs are best at epoch 3 and 2
(0.496 and 0.488, after about 800 and 500 of their 7,860 steps) and lower at their last
epoch, while the equal-epochs runs are best at epochs 30 and 29, their last. On 5% of the
beats, the steps beyond about 800 bought nothing, and the gap between the two settings is
not the cost of the steps.

**What every beat buys over 5%, at equal steps.** At equal steps the share moves the
data alone, and the table of every share against 5% reads it: from 5% to every beat the
route arm gains +6.9 points (se 1.1) and the control +10.2 (se 1.0). The last epoch of
the 5% runs is not their best, so the comparison holds at the best epochs too: there,
from 5% to every beat, the route arm gains 7.0 points (0.496 to 0.566) and the control
9.6 (0.488 to 0.584). Of the steps between, the route arm gains most from 0.05 to 0.2,
+5.0 (se 1.6), and the control from 0.2 to every beat (below). No point sits at the
always-N floor: the lowest mean is 0.472.

**Whether data still pays at the top of the curve, these runs leave open.** The last
step, from 0.2 to every beat, is where the curve would flatten. At equal steps the
control gains +6.2 points there (se 2.6) and the route arm +1.9. At equal epochs the
control's last step is +5.3 (se 1.8), but at equal epochs the steps grow with the data
(1,590 to 7,860), so that step buys data and steps together. So these runs say that
every beat is far better than 5% for both arms, and leave open how much the control
still gains between a fifth of the beats and all of them. There are no more
training beats than 1.0: a curve that still rises there is a finding about the data, not
a range to search past.

**Routing did not beat its control on this curve.** At 5% and equal epochs the route arm
is 3.0 points above its control (se 1.2), but that run was still climbing; at equal
steps the same share gives +1.3. At 0.2 the route arm is ahead by 2.4 and 2.2 points and
at every beat 2.1 behind: the sign flips
between 0.2 and 1.0 in both settings. Whether routing does better on little data is a
bet for Part 2, not a finding of this curve.

## E4: a search space from your curves

Part 2 asks you for a search space. A space is a bet on where the answer lies: a trial
spent where the curve is flat, or on an axis that changes nothing, is a trial not spent
where the curve still climbs. E2 and E3 measured parameters and data. The third axis,
compute, counted here as the examples a run sees, is epochs × training beats; on every
beat that is the epochs, and those are already in E2's runs: every run scored the
validation patients after every epoch. The cell below draws that curve, the mean over
the repeats per width and arm, and prints the last eleven epochs with the spread of the
last one.

The score of one epoch is noisy: from one epoch to the next, the mean of three runs can
move by a few hundredths. E2 and E3 compare the last epoch, as a study does. The second
table is the check on that: per width, route minus matched at the last epoch and
averaged over the last ten epochs. Where the two agree, the last epoch is a fair reading;
where they part, read the average.

In [ ]:
epochs = np.arange(1, EPOCHS + 1)
# one panel per width: every run's own curve thin, the mean of the runs thick, so you
# see whether the runs of a point have settled and whether one seed is still climbing
fig, panels = plt.subplots(
    1, len(WIDTHS), figsize=(3.4 * len(WIDTHS), 3.4), sharey=True, squeeze=False
)
rows = {}
for ax, w in zip(panels[0], WIDTHS, strict=True):
    for name, color in [("route", "tab:orange"), ("matched", "tab:blue")]:
        per_epoch = np.array(e2[name, w].curves)  # (repeats, epochs)
        for run in per_epoch:
            ax.plot(epochs, run, color=color, lw=0.8, alpha=0.45)
        ax.plot(epochs, per_epoch.mean(0), color=color, lw=2.2, label=name)
        rows[name, w] = {
            **{e: m for e, m in zip(epochs, per_epoch.mean(0), strict=True)},
            "std at the last epoch": per_epoch[:, -1].std(ddof=1),
        }
    ax.set_title(f"width {w}", loc="left", fontsize=10)
    ax.set_xlabel("epoch")
    ax.spines[["top", "right"]].set_visible(False)
panels[0, 0].set_ylabel(f"{OBJECTIVE} on the validation patients")
panels[0, -1].legend(fontsize=8, title=f"thin: each of {REPEATS} runs")
plt.suptitle(
    "the curve over compute: E2's runs after every epoch",
    x=0.02,
    ha="left",
    fontsize=10,
)
plt.tight_layout()
plt.show()
by_epoch = pd.DataFrame(rows).T
by_epoch.index.names = ["arm", "width"]
display(by_epoch.iloc[:, -12:].round(3))

# route against matched at the last epoch, and averaged over the last ten
last = min(10, EPOCHS)
check = pd.DataFrame(
    {
        w: {
            "at the last epoch (pt)": 100
            * (e2["route", w].values.mean() - e2["matched", w].values.mean()),
            f"mean of the last {last} epochs (pt)": 100
            * (
                np.array(e2["route", w].curves)[:, -last:].mean()
                - np.array(e2["matched", w].curves)[:, -last:].mean()
            ),
            "largest move of a mean between two epochs": max(
                np.abs(np.diff(np.array(e2[name, w].curves).mean(0)[-last:])).max(
                    initial=0.0
                )
                for name in ["route", "matched"]
            ),
        }
        for w in WIDTHS
    }
).T
check.index.name = "width"
print("route - matched, per width")
display(check.round(3))

Nothing in Part 1 is seeded, so your numbers will differ; the readings below are of the run they were written from.

**By epoch 20 the curves have flattened.** Over epochs 21 to 30, the mean of the last
five epochs of every line is within 0.01 of the mean of the five before (from -0.010
for the route arm at width 64 to +0.007 for the control at width 8): no line rises by
more than 0.01 per five epochs. The table starts at epoch 20, so it does not say where
before that each line reached its level; the figure is too crowded to read that off. So
the comparisons of E2 and E3 at 30 epochs are between arms that have stopped climbing,
on every beat; a search that draws fewer than about 20 epochs may compare arms before
they stopped, and partly on how fast they learn.

**The last epoch is noisy, and at width 64 it sets the size of the gap.** The mean of
three runs moves by up to 0.011 to 0.025 between two of the last ten epochs. Route minus
matched at the last epoch and averaged over the last ten: -0.6 and -1.0 points at width
8, -0.1 and 0.0 at 16, -2.1 and -1.7 at 32, -3.5 and -1.3 at 64. At width 64 the route
arm drops at its last epoch (0.558 to 0.545) while its control rises. The table shows
that the size of E2's largest gap depends on the epoch it is read at. What holds in both
readings is the sign: the route arm is below its control or level at every width, never
above it.

**Before you read on, write your own space down**, from your three curves, as you will
put it in `SPACE` of [`exercises/study.py`](exercises/study.py). Five questions:

1. **Size.** Over which widths did a step of the curve over parameters (E2) pay, and by
   how many points beside its standard error? A stretch whose steps are about the size
   of their standard errors is still open, not flat. Search where it paid; where no step
   clearly paid, as in the run this text was written from, the open stretch is what
   there is to search, and say whether your search spends its trials there or fixes the
   width. Depth is a second way to add parameters, which Part 1 did not measure: search
   it or fix it, and say why. Depth also sets where the route stage sits: on the
   shortened beat, 93, 46, 23 or 11 positions at depth 1 to 4.
2. **Data.** The same for `train_fraction`, from the curve over data (E3).
3. **Compute.** From the curve above: after how many epochs does it flatten? Is `epochs`
   an axis worth searching, or one to fix at a value past the bend? Remember that a
   space that searches `train_fraction` at fixed `epochs` moves the examples seen with
   it, unless it sets `equal_steps`.
4. **Fixed.** Which axes do you hold at one value, and at which? Every axis you fix is
   one the trials do not have to cover, so the others get more of them.
5. **Controls.** The route arm against what? `matched` holds the parameters, the data
   and the epochs equal and takes the router away. Dense routing (`MoE`) runs every
   expert at every position, so its arithmetic per position is about that of `matched`
   too. Sparse routing runs one expert per position, so it computes less than its
   control at the same parameters. Say what your comparison holds equal and what it
   does not, so that a gain can be put on routing and not on more weights or more
   compute. Without `matched` in the trials, a gain of `route` says only that
   parameters helped.

**When an axis is at its limit.** An axis that still climbs at the end of its range has
no range past it to search: there are no more training beats than 1.0, and epochs past
`EPOCHS` cost minutes you may not have. That is a finding about the budget. For a study
that looks for the best design, fix the axis at its limit (every beat; the most epochs
your minutes allow) and say in your page that the curve had not flattened there. For a
study whose question is the curve itself, such as "does routing pay with less data?",
search the axis below its limit: then the trials measure how the difference moves along
it, and that is the answer, not a winner. Say which of the two your study is.

## An example space, and what it bought

The space below was set before any of these curves existed: every capacity axis over a
wide range. It is an example to read against your own, not the answer. The class runs it
once for two more reasons: its trials measure what one trial costs on your machine,
which is what Part 2's budget is computed from, and the class's runs of it are read
together (the end of this notebook).

It holds its control in the right way: `route` is on in every trial and `matched` is
searched, and every draw is **paired**. A trial draws its width, depth, share and epochs;
the next trial is its twin, the same draw with `matched` flipped, built and trained from
the same seed: the same starting weights in every layer but the stage, and the beats in
the same order. Within a pair the two arms differ in the stage and in the noise that
training adds to both, so the difference `route - matched` of a pair is the effect of
routing at that draw plus that noise. That is the control next to the arm that the
search needs to tell the effect of routing apart from the effect of more weights, data
or epochs.

| axis | values | what it turns |
|---|---|---|
| `width` | 8 to 64 | parameters: channels per block |
| `depth` | 1 to 4 | parameters: blocks |
| `train_fraction` | 0.05 to 1.0 | data |
| `epochs` | a quarter of `EPOCHS` to `EPOCHS` | compute |
| `route` | on | the route stage in every trial |
| `matched` | off, on | the pair: the route stage, and its control in its place |

The search samples at random and prunes nothing. Random, not the TPE of notebook 3.3,
because this search is a map of the space, not a hunt for its best point: TPE puts its
later trials where the score was high, so the curve is dense there and thin elsewhere,
and the groups of the readout below would hold the pairs TPE liked. A random draw covers
every part of every axis alike. No pruner, because a pruner stops a trial on its first
epochs, and a curve over compute needs every trial to run the epochs it drew.

**How the pairs are read.** Lesson 2's standard error is for two independent arms:
`sqrt(std_route² / n + std_matched² / n)`. Here the arms are not independent: the two
trials of a pair share their draw and their seed. So the readout takes the difference
within every pair first, and its standard error is the standard deviation of those
differences over the square root of the number of pairs. Everything the two trials of a
pair share, a width of 8 or of 64, a share of 5% or of every beat, falls out of the
difference. The table prints both, `se` (paired) and `se unpaired` (lesson 2's formula
on the same trials), so you can see what pairing bought.

In [ ]:
EXAMPLE_SPACE = Space(
    {
        "dataset": "mitbih",
        "width": Int(8, 64, log=True),
        "depth": Int(1, 4),
        "train_fraction": Float(0.05, 1.0, log=True),
        "epochs": Int(max(1, EPOCHS // 4), EPOCHS),
        "route": True,
        "matched": Choice([False, True]),
        "n_experts": 4,
        "routing": "dense",
        "balance": 0.0,
        "parallel": False,
        "fold": False,
        "skip": False,
        "excite": False,
        "state": False,
        "gate": False,
        "class_weight": 0.0,
        "lr": 1e-3,
    }
)
example = search(EXAMPLE_SPACE, "lesson3-notebook")
# the trials into RESULTS, next to E2 and E3, and read back from there
stored = json.loads(RESULTS.read_text())
stored["e4"] = json.loads(trial_rows(example).to_json(orient="split"))
RESULTS.write_text(json.dumps(stored))
table = pd.read_json(io.StringIO(json.dumps(stored["e4"])), orient="split", dtype=False)
both = pairs(table)
print(f"{len(table)} trials, {len(both)} complete pairs")

axes = ["n_params", "train_fraction", "train_examples"]
fig, panels = plt.subplots(1, len(axes), figsize=(15, 3.4), sharey=True)
for ax, axis in zip(panels, axes, strict=True):
    ax.scatter(both[axis], both["route - matched"], color="tab:purple", alpha=0.8)
    ax.axhline(0, color="gray", lw=1)
    ax.set_xscale("log")
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel(axis)
    ax.spines[["top", "right"]].set_visible(False)
panels[0].set_ylabel(f"route - matched, {OBJECTIVE}")
plt.suptitle(
    "every pair of the search: route minus its control on the same draw and seed",
    x=0.02,
    ha="left",
)
plt.tight_layout()
plt.show()
for axis in axes:
    display(
        paired_curve(both, axis).style.format(
            "{:.3f}",
            subset=["route", "matched", "route - matched", "se", "se unpaired"],
            na_rep="-",
        )
    )
# the spread of the differences within the pairs
PAIR_SPREAD = both["route - matched"].std(ddof=1) if len(both) > 1 else float("nan")
if len(both) > 1:
    print(
        f"the differences within the pairs: std {PAIR_SPREAD:.3f}; the route arm's "
        f"trials {both['route'].std(ddof=1):.3f}, the control's "
        f"{both['matched'].std(ddof=1):.3f}; correlation of the two arms over the "
        f"pairs {both['route'].corr(both['matched']):.2f}"
    )
print("the route arm's trials, the most concentrated router first:")
display(use_table(table).round(3))

In [ ]:
def read_pairs(group: pd.DataFrame) -> dict:
    # route against its control over the pairs of `group`: the mean difference, and the
    # paired and the unpaired standard error beside it; a spread needs two pairs, so
    # fewer give none
    diff = group["route - matched"]
    if len(diff) < 2:
        return {"pairs": len(diff), "route - matched (pt)": 100 * diff.mean()}
    se = diff.std(ddof=1) / np.sqrt(len(diff))
    unpaired = np.sqrt(
        (group["route"].var(ddof=1) + group["matched"].var(ddof=1)) / len(diff)
    )
    return {
        "pairs": len(diff),
        "route - matched (pt)": 100 * diff.mean(),
        "se (pt)": 100 * se,
        "se unpaired (pt)": 100 * unpaired,
    }


# a pair in which an arm scored a third of the macro recall, one class out of three,
# is a pair in which that arm learned nothing: it measures a failure to train
FLOOR = 1 / 3 + 1e-6
floor = (both["route"] < FLOOR) | (both["matched"] < FLOOR)
summary = pd.DataFrame(
    {
        "every pair": read_pairs(both),
        "without the floor pairs": read_pairs(both[~floor]),
    }
).T
signed = [c for c in ["route - matched (pt)"] if c in summary]
spreads = [c for c in ["se (pt)", "se unpaired (pt)"] if c in summary]
display(
    summary.style.format("{:+.1f}", subset=signed, na_rep="-").format(
        "{:.2f}", subset=spreads, na_rep="-"
    )
)
below = int((both["route - matched"] < 0).sum())
above = int((both["route - matched"] > 0).sum())
print(f"{below} pairs below zero, {len(both) - below - above} on it, {above} above")
print("the floor pairs:")
display(
    both.loc[floor, ["route", "matched", "width", "depth", "train_fraction"]].round(3)
)

# per group of `paired_curve`: how many of its pairs are floor pairs
rows = []
for axis in axes:
    for label, members in both.groupby(groups(both, axis), observed=True):
        rows.append(
            {
                "axis": axis,
                "group": label,
                "pairs": len(members),
                "floor pairs": int(floor[members.index].sum()),
            }
        )
display(pd.DataFrame(rows).set_index(["axis", "group"]))

Nothing in Part 1 is seeded, so your numbers will differ; the readings below are of the run they were written from.

**All twelve pairs together: the route arm is 1.8 points below its control**, with a
paired standard error of 0.6 points. Nine pairs have the route arm below its control,
one is level and two are above (the scatter). That is the same direction as E2, on every
beat at 30 epochs, where the route arm was 0.1 to 3.5 points below at every width: two
experiments that lean the same way.

**Two pairs measure a failure to train.** In two pairs an arm learned nothing: a macro
recall of 0.333, which is always N. In the first only the route arm did (0.333 against
0.369); in the second both did (0.333 and 0.333, a difference of 0 that says nothing
about routing). Both are small models, width 8 and 9 at depth 3, on 8 and 14% of the
beats. A pair in which an arm learned nothing measures whether a model trains at all,
not what routing does once it trains. Without them the ten other pairs give the same
-1.8 points (se 0.7).

**No group shows routing ahead.** The 24 trials made 12 complete pairs, 4 in every
group. In every group the route arm is below its control on average, by 0.8 to 2.8
points, with paired standard errors of 0.8 to 1.4 points. The largest gaps sit at the
low end of every axis: the smallest models (-2.8 points, se 1.3), the smallest shares
(-2.2, se 1.0) and the fewest examples seen (-2.5, se 0.9). These three groups are not
three findings: they hold the same two pairs in which an arm learned nothing, plus two
others each. Read them as one lean of the low end, carried in part by two failures to
train.

**Read together with E2**, nothing here shows routing paying on the heartbeats: two
experiments put it level with, or a point or two below, the same parameters spent on a
plain path, beside standard errors of about a point.

**Pairing is what made the groups readable.** The unpaired standard error, lesson 2's
formula on the same trials, is 0.067 to 0.076 in the three low groups, against 0.009 to
0.013 paired: within a group the draws still differ in width, depth, share and epochs,
and the pairs take that out. Over all pairs the route arm's trials spread by 0.075 and
the control's by 0.074, but their differences by 0.022: the two arms of a pair move
together (a correlation of 0.96). Over all twelve pairs the unpaired standard error is
3.0 points against 0.6 paired: beside the unpaired one, the same -1.8 points is lost in
what the draws differ by. In
the middle group of `train_examples` the unpaired standard error (0.012) is a little
below the paired one (0.014): its draws differ, but their scores are close, and a
standard error from four pairs (three degrees of freedom) is itself noisy, so 0.012
against 0.014 is no difference between the two.

**The routers.** All twelve route trials are spread: every expert has a peak of 5% or
more. The most concentrated, width 8 and depth 3 on a third of the beats, gives one
expert a mean weight of 0.874. Expert 1 of trial 8 has a peak that prints as 0.050 in its
`expert_peak` (in `optuna-dashboard`) and counts as in use, while the expert of E2's
width-8 run with the same printed peak counts as idle: the count compares the unrounded
peak with 5%, and both sit on the fourth decimal, so neither label says anything about
its router.

Hold the example against the five answers you wrote down:

- **Ranges.** Where does this space spend its pairs that your curves of E2, E3 and E4
  had already answered, and where does it leave out a stretch they left open?
- **Pairs per group.** How many pairs does each group get, and how large is its `se`
  beside the differences you read? `se` is the spread of the differences over the
  square root of the pairs.
- **Control.** Compare `se` with `se unpaired`. Which of your axes does your question
  need the pairs along?

Your answers are your `SPACE` in Part 2. Your runs of Part 1 are now in
`lesson3-notebook` and `lesson3-notebook-runs`.

# Part 2: your own study

The loop is the one of lessons 1 and 2, on lesson 3's design; `docs/your-study.md` walks
its nine steps. You work in [`exercises/`](exercises/):

| file | what it is | what you do with it |
|---|---|---|
| [`exercises/designs.py`](exercises/designs.py) | `Beats`, one typed method per stage; lesson 2's blocks and `route` and `matched` are options | read it with atlas; add a stage when your idea needs another block, and a control of the same parameter count for it |
| [`exercises/study.py`](exercises/study.py) | six settings at the top (`STUDY`, `IDEA`, `EXPECTS`, `OBJECTIVE`, `SPACE`, `MINUTES`) and the code that runs the study | fill in the settings, then `make study LESSON=3` |
| [`exercises/harness.py`](exercises/harness.py) | the heartbeats split by patient, `train_fraction`, the trainer, the linear baseline, the curves and the control | nothing: it is given |

`make study LESSON=3` needs Part 1 to have run on your machine: it reads the cost of a
trial from Part 1's search.

What the study does, in order: it measures the linear baseline, searches your `SPACE` in
pairs as E4 does (every draw of the route stage followed by its twin with `matched`
flipped, on the same seed), reruns the winner and compares it with the baseline in
points (as in lesson 2), then prints the paired curves along every capacity axis it
moved, the spread of the differences within the pairs, and the use of the experts of its
route trials; last it runs the winner's config with the route stage and with its
control, each `REPEATS` times, prints the use of the experts of the route runs, and the
two arms against each other in points, each beside its spread. Every trial and run with
a router carries the use of its experts as user attrs, Optuna's store of key-value pairs
per trial, which `make optuna-dashboard` shows under "User Attributes" (`expert_share`,
`expert_weight`, `expert_peak`, `top_expert_share`, `top_expert_weight`,
`experts_in_use`), so `make optuna-dashboard` shows it next to its score.

**Your study searches what its question needs.** The route stage is paired with its
control in every study. Beyond that there are two kinds of study, and your curves say
which axes each one searches:

- **A study for the best design** searches the axes that still move. An axis at its
  limit is fixed there (every beat, the epochs past the bend), and the search spends its
  trials on the axis left, say width over the stretch E2 left open.
- **A study of a curve** searches the axis whose curve is the question, below its limit
  too. For "does routing pay with less data?": `train_fraction` from 0.05 to 1.0 with
  `equal_steps`, the width fixed at one value, `epochs` fixed past the bend. The
  readout is then the paired difference per group along `train_fraction`, not a winner.

Which kind, which axes, over which ranges, what is fixed and which control is yours to
decide: the five questions of E4 are the argument, and the example space is only an
example.

## The open questions

Every question below is one bet a study can test; the class spreads over them, so that
the next lesson opens on a map of what capacity bought, each with its spread, not a
leaderboard. A bet that did not pay is a result on that map.

- **Does routing pay with more data, or with less?** A router has to learn which expert
  takes which position. Does the gap between `route` and `matched` change sign along
  `train_fraction`?
- **Do more experts buy anything?** `n_experts` sets how many paths the router chooses
  between, and the control grows with it. At a fixed width, does a curve over
  `n_experts` flatten, and where?
- **Sparse against dense.** `routing="sparse"` runs one expert per position (notebook
  3.1's `SparseMoE`): it trains as many parameters and uses fewer per position. Does it
  lose against the dense mixture at the same `n_params`, and does its router collapse
  onto one expert (`experts_in_use` of 1)?
- **Does a balanced router pay?** With `routing="sparse"`, `balance` adds the
  load-balancing loss of the lesson 3 slides to the training loss, only while training
  (Fedus et al.,
  2021, use 0.01). It is there against router collapse: it grows as the router
  concentrates the positions on few experts. On the heartbeats, does a balanced router
  spread its positions, and does it then beat its control, or the collapsed router?
- **Does S need parameters, data or compute?** S is rare. Read the recall of S of your
  trials along each capacity axis: which one moves it?
- **Is the problem bound on compute?** `epochs` is an axis too. Is a small model trained
  longer worth as much as a large one trained briefly, at the same seconds?
- **Does a lesson 2 block change the curve?** `skip`, `state`, `gate` and the others are
  still options. Does one of them shift the curve over parameters, against a control of
  the same size?

A question of your own is welcome, as long as it is a bet you can state before you run.

## The steps

Run every command in a terminal, from the root of the repository.

1. **Design.** Read the design in atlas:

   ```bash
   atlas-code-nightly view notebooks/3_scaling_and_search/exercises
   ```

2. **Build.** Change the design if your question needs a block it does not have, then
   check that every combination builds and that the route stage and its control have the
   same parameters:

   ```bash
   uv run notebooks/3_scaling_and_search/exercises/designs.py
   ```

3. **Baseline.** Nothing to do: `make study LESSON=3` measures the linear baseline before
   it searches.
4. **Budget and space.** `STUDY` is your first name and what it studies:
   `"sam-experts-on-little-data"`. `IDEA` says what you investigate in your own words;
   your first idea builds on Part 1 (`builds_on=Idea.handed_out(3).name`), a later one on
   an idea of the class, by its name. `SPACE` pairs the route stage with its control and
   searches the axes your question needs, over the ranges your answers to E4's five
   questions give, and fixes the rest; every option of the design is an axis, searched
   or fixed. `MINUTES` is your budget; the cell below shows what it buys.

   **Your hypothesis, before you run.** `EXPECTS` is one line, written now: does `route`
   beat `matched`, by how many points, and where on your curves; will its router
   spread the positions over the experts or collapse onto one; and why you expect it.
   It is stored with the study and never checked, so the result can surprise you.
5. **Search:** `make study LESSON=3`. It searches, reruns the winner, prints the paired
   curves, the use of the experts of the route trials, and the route stage against its
   control in points.
6. **Read** it in `make optuna-dashboard`: per axis, what it did to the macro recall,
   the `n_params` of every trial, and the recall of S of your best trials (every trial
   logs its recall per class in the user attr `metrics`).

   **The experts, after the run.** Read `expert_share`, `expert_weight` and
   `expert_peak` of your route trials, and of the route runs at the winner (in the
   study's `-runs`): was the router spread, partly collapsed or collapsed
   (`experts_in_use`: the experts whose peak is 5% or more, of the share for sparse
   routing, of the weight for dense)? Did the collapsed trials score like their
   control? At the winner, the comparison favours the arm of the winning trial: the
   search picked that config because that arm scored highest with it on the validation
   patients, and the rerun scores both arms on the same patients. Part of what made the
   config win is luck on those patients that suits the winning arm, and the rerun keeps
   it. So read the arms at the winner as a check on the winner, not as evidence about
   routing; the paired curves over all your pairs are that evidence. If you searched `balance`, what
   did it do to the use, and to the macro recall? Hold the answers against your
   `EXPECTS`.
7. **Explain:** `make page LESSON=3` builds your page of the lesson from your store; its
   step on the page says what to fill in.
8. **Share:** `make share LESSON=3`.
9. **Build on:** `make dashboard LESSON=3` opens the class page and `optuna-dashboard`
   on the class store, built from what the class handed in. The class page shows the
   ideas of the class as a graph of what builds on what, each idea's winner against its
   own baseline, and the handed-out search pooled over the class, axis by axis. In
   `optuna-dashboard` the handed-out search is one study, `lesson3-notebook`, with the
   trials of every student, and every study of a classmate is there as
   `<study>@<student>`, every trial with its user attrs: its `pair`, the use of its
   experts, its `metrics`. Your next idea builds on one of the class.

In [ ]:
MINUTES = 15  # the time you give your study on this machine
# put your own SPACE here; this one is the example space on every beat at EPOCHS epochs
YOUR_SPACE = Space({**EXAMPLE_SPACE, "train_fraction": 1.0, "epochs": EPOCHS})
PART1 = StudyLog.load(name="lesson3-notebook")


def trials_in(minutes: float, space: Space) -> int:
    # the trials of `space` that fit in `minutes`, once the baseline and the winner's
    # rerun (REPEATS runs each) are set aside
    return Budget.at(
        minutes=minutes,
        seconds_per_trial=seconds_per_trial(PART1, space),
        measured_on=PART1.name,
        epochs=EPOCHS,
        repeats=REPEATS,
    ).n_trials


minutes = np.arange(1, 61)
fig, ax = plt.subplots(figsize=(8, 3.4))
for label, space, color in [
    ("your space", YOUR_SPACE, "tab:blue"),
    ("the example space", EXAMPLE_SPACE, "tab:gray"),
]:
    cost = seconds_per_trial(PART1, space)
    ax.plot(
        minutes,
        [trials_in(m, space) for m in minutes],
        color=color,
        label=f"{label}: {cost:.0f}s a trial",
    )
ax.scatter(
    [MINUTES], [trials_in(MINUTES, YOUR_SPACE)], color="tab:orange", s=80, zorder=3
)
ax.set_xlabel("minutes you give the study")
ax.set_ylabel("trials that fit")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title(
    "trials that fit, from what a trial cost in the search of Part 1, per epoch and per "
    "training example",
    loc="left",
    fontsize=10,
)
plt.show()

trials = trials_in(MINUTES, YOUR_SPACE)
twinless = " (the odd trial has no twin)" if trials % 2 else ""
print(
    f"{MINUTES} minutes: {trials} trials of your space, {trials // 2} pairs{twinless}, "
    f"{len(YOUR_SPACE.searched())} searched axes"
)

Every trial of the plot is one arm of a pair, so the pairs are half the trials; an odd
number of trials leaves the last one without its twin. The lines start at zero: the
study first measures the linear baseline and, after the search, reruns the winner
`REPEATS` times, and the plot sets those minutes aside before it counts a trial. What a
trial costs depends on its space: a trial on every beat at `EPOCHS` epochs costs more
than the example search's fit at its mean draw, whose share and epochs are smaller, and
a trial on a small share or few epochs costs less. The plot reads both from
the search of Part 1 and draws your space next to the example. The study also runs the
route stage and its control at the winner, `REPEATS` runs each, after the trials this
plot counts: give it a few minutes more than the trials alone would need.

**Pairs read together.** The handed-out search of Part 1 is pooled already: every
student runs it, and the class store holds it as one study. Studies that search the
same space, with the same `builds_on`, hand in pairs that can be read together too:
agree on a space with classmates who take the same open question, and name the same idea
in `builds_on`.

## Hand in your runs before the next lesson

```bash
make share LESSON=3
```

It hands in Part 1 (`lesson3-notebook`, `lesson3-notebook-runs`) and every study of
yours (`lesson3-study-*`), each run with the design script it ran. Name the ideas of
classmates you built on in `builds_on`; the class page shows who built on whom.

## The class's pairs

Once the class has handed in, `make dashboard LESSON=3` builds the class store,
`artefacts/optuna/class.db`. The cell below reads the pairs of the handed-out search of
everyone in it, along the same three axes as E4; `class_pairs("<study>@<student>", ...)`
reads the studies of classmates who searched one space together.

In [ ]:
pooled = class_pairs()  # the handed-out search of the whole class
if pooled.empty:
    # the store's path from the repository root
    where = Path(*CLASS_DB.parts[-3:])
    print(
        f"no class store at {where} yet: run `make dashboard LESSON=3` once the class has handed in"
    )
else:
    print(f"{len(pooled)} pairs of the class")
    for axis in axes:
        display(paired_curve(pooled, axis).round(3))